# 01 — L0 toy ground state (Hubbard dimer)

**Goal:** VQE on the 2-site Hubbard dimer at U=4, t=1, eps=0.

**Validation layers** (from spec §5):
1. Energy match against ED (|ΔE| < 1 mHa)
2. State overlap ≥ 0.99
3. Ansatz expressivity (max overlap when classically optimized) ≥ 0.99
4. Multi-start spread across 8 seeded starts < 5 mHa
5. Observable agreement (n_total, S², double occupancy) within 5%

All five layers must pass. Each is reported with numerical margin.


In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import siam_vqe as svqe
from siam_vqe import (
    check_ansatz_expressivity, check_energy_match, check_multistart_spread,
    check_observable_agreement, check_state_overlap,
    efficient_su2_ansatz, exact_diag, hubbard_dimer, observables_dimer,
    plot_convergence, run_vqe, run_vqe_multistart, to_qubit_op,
)
print(f'siam_vqe {svqe.__version__}')


## 1. Set up the L0 Hubbard dimer


In [ ]:
U, t, eps = 4.0, 1.0, 0.0
num_particles = (1, 1)  # half-filling, Sz=0

fop = hubbard_dimer(U=U, t=t, eps=eps)
pop = to_qubit_op(fop, scheme='parity_tapered', num_particles=num_particles)
print(f'Hubbard dimer (U={U}, t={t}, eps={eps})')
print(f'qubits (parity_tapered): {pop.num_qubits}')
print(f'Pauli terms: {len(pop)}')


## 2. ED reference


In [ ]:
ed = exact_diag(pop, k=4)
print(f'ED ground state energy E0 = {ed.energies[0]:.8f}')
print(f'lowest 4 levels: {np.round(ed.energies, 6)}')
# Analytic check:
import math
e_analytic = 2 * eps + U / 2 - math.sqrt((U / 2) ** 2 + 4 * t ** 2)
print(f'analytic GS: {e_analytic:.8f}')
print(f'|ED - analytic| = {abs(ed.energies[0] - e_analytic):.2e}')


## 3. VQE run (EfficientSU2 + COBYLA)


In [ ]:
circuit, x0 = efficient_su2_ansatz(num_qubits=pop.num_qubits, reps=3, seed=20260524)
print(f'ansatz: EfficientSU2 reps=3, {circuit.num_parameters} parameters')
result = run_vqe(pop, circuit, x0, optimizer='COBYLA', maxiter=600)
print(f'VQE energy: {result.energy:.8f}')
print(f'walltime: {result.walltime_s:.2f}s, evals: {len(result.history)}')


### Convergence plot


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_convergence(result, ed_energy=float(ed.energies[0]), ax=ax)
plt.tight_layout()
plt.show()


## 4. Validation layers


In [ ]:
# Layer 1: energy match
r_energy = check_energy_match(result.energy, float(ed.energies[0]), tol_hartree=1e-3)
print(f'Layer 1 (energy match): {"PASS" if r_energy.passed else "FAIL"} - dE = {r_energy.delta:.2e}')


In [ ]:
# Layer 2: state overlap
r_overlap = check_state_overlap(result, ed, threshold=0.99)
print(f'Layer 2 (state overlap): {"PASS" if r_overlap.passed else "FAIL"} - overlap = {r_overlap.overlap:.6f}')


In [ ]:
# Layer 3: ansatz expressivity (classical max-overlap probe)
r_express = check_ansatz_expressivity(
    circuit=result.circuit, target_state=ed.vectors[:, 0],
    threshold=0.99, seed=20260524, maxiter=300,
)
print(f'Layer 3 (ansatz expressivity): {"PASS" if r_express.passed else "FAIL"} - max overlap = {r_express.max_overlap:.6f}')


In [ ]:
# Layer 4: multi-start spread
multistart = run_vqe_multistart(
    pop, circuit, n_starts=8, ansatz_factory_seed_base=1000,
    ansatz_num_qubits=pop.num_qubits, ansatz_reps=3,
    optimizer='COBYLA', maxiter=400,
)
r_spread = check_multistart_spread(multistart, max_spread_hartree=5e-3)
print(f'Layer 4 (multi-start spread): {"PASS" if r_spread.passed else "FAIL"}')
print(f'  best={r_spread.best:.6f}, median={r_spread.median:.6f}, spread={r_spread.spread:.2e}')
energies = [r.energy for r in multistart]
print('  per-start energies:', np.round(energies, 6))


In [ ]:
# Layer 5: observable agreement
obs_fops = observables_dimer()
obs_reports = {}
for name in ['n_total', 'S2', 'double_occ']:
    obs_pop = to_qubit_op(obs_fops[name], scheme='parity_tapered', num_particles=num_particles)
    rep = check_observable_agreement(result, ed, observable=obs_pop, rel_tol=0.05, name=name)
    obs_reports[name] = rep
    print(f'Layer 5 ({name}): {"PASS" if rep.passed else "FAIL"} - VQE={rep.vqe_value:.4f}, ED={rep.ed_value:.4f}, rel_err={rep.rel_error:.2%}')


## 5. Persist results


In [ ]:
out = {
    'system': {'U': U, 't': t, 'eps': eps, 'num_particles': list(num_particles)},
    'ansatz': {'name': 'EfficientSU2', 'reps': 3, 'num_parameters': int(circuit.num_parameters)},
    'optimizer': result.optimizer_name,
    'maxiter': 600,
    'vqe_energy': float(result.energy),
    'ed_energy': float(ed.energies[0]),
    'validations': {
        'layer_1_energy_match': {'passed': r_energy.passed, 'delta': float(r_energy.delta), 'tol': r_energy.tol_hartree},
        'layer_2_state_overlap': {'passed': r_overlap.passed, 'overlap': float(r_overlap.overlap), 'threshold': r_overlap.threshold},
        'layer_3_ansatz_expressivity': {'passed': r_express.passed, 'max_overlap': float(r_express.max_overlap), 'threshold': r_express.threshold},
        'layer_4_multistart_spread': {'passed': r_spread.passed, 'best': float(r_spread.best), 'median': float(r_spread.median), 'spread': float(r_spread.spread)},
        'layer_5_observables': {name: {'passed': rep.passed, 'vqe': float(rep.vqe_value), 'ed': float(rep.ed_value), 'rel_err': float(rep.rel_error)} for name, rep in obs_reports.items()},
    },
}
out_path = Path('01_L0_vqe_result.json')
out_path.write_text(json.dumps(out, indent=2))
print(f'wrote {out_path}')
all_pass = r_energy.passed and r_overlap.passed and r_express.passed and r_spread.passed and all(r.passed for r in obs_reports.values())
print(f'\nALL FIVE LAYERS PASS: {all_pass}')
